# Lesson 01 - Pengantar Agen AI

Selamat datang di pelajaran pertama dalam kursus **Agen AI untuk Pemula**!

Sebuah **agen AI** adalah program yang menggunakan model bahasa besar (LLM) sebagai mesin penalarannya dan dapat mengambil *tindakan* di dunia nyata — memanggil API, melakukan kueri basis data, atau menjalankan kode — untuk mencapai tujuan atas nama pengguna.

Di notebook ini Anda akan membangun agen pertama Anda: sebuah **Agen Perjalanan** yang merekomendasikan destinasi liburan. Sepanjang jalan Anda akan belajar bagaimana:

1. Terhubung ke Azure AI Foundry Agent Service menggunakan **Microsoft Agent Framework**.
2. Memberikan agen sebuah **alat** — fungsi Python biasa yang dapat dipanggilnya.
3. Menjalankan agen dan memeriksa responsnya.
4. Mengalirkan respons agen token demi token.


## Setup

Sebelum menjalankan notebook ini, pastikan Anda memiliki:

1. **Proyek Azure AI Foundry** dengan model chat yang telah diterapkan (misalnya `gpt-4o-mini`).
2. **Masuk dengan Azure CLI** — jalankan `az login` di terminal Anda.
3. **Atur variabel lingkungan yang diperlukan:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint proyek Azure AI Foundry Anda.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model yang telah Anda terapkan.

Selanjutnya, sel di bawah ini menginstal paket Python yang Anda butuhkan.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Membuat Agen Pertama Anda

Sebuah agen membutuhkan dua hal:

- **Instruksi** yang memberitahunya *siapa dia* dan *bagaimana cara berperilaku* (prompt sistem).
- **Alat** — fungsi Python yang dihias dengan `@tool` yang dapat dipanggil agen untuk mengambil informasi atau melakukan tindakan.

Di bawah ini kami mendefinisikan alat sederhana yang mengembalikan daftar tujuan liburan populer. Agen akan menggunakan alat ini ketika pengguna meminta rekomendasi perjalanan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Untuk pengalaman yang lebih interaktif, Anda dapat **stream** respons agen. Alih-alih menunggu balasan lengkap, agen mengirimkan potongan teks saat mereka dihasilkan. Ini sangat berguna dalam antarmuka obrolan di mana Anda ingin menampilkan keluaran secara real-time.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Ringkasan

Dalam pelajaran ini Anda mempelajari cara untuk:

- **Membuat penyedia** yang terhubung ke Azure AI Foundry Agent Service melalui `AzureAIProjectAgentProvider`.
- **Mendefinisikan alat** menggunakan dekorator `@tool` sehingga agen dapat memanggil fungsi Python Anda.
- **Menjalankan agen** dengan pesan pengguna dan mencetak responsnya.
- **Streaming respons** untuk output waktu nyata.

Dalam pelajaran berikutnya kita akan menjelajahi kerangka kerja agen secara lebih mendalam dan mempelajari cara memberikan alat yang lebih kuat dan kemampuan penalaran multi-langkah kepada agen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
